In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import re


# Hate

In [3]:
hate_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Raw/hindi_hate_dataset_20k.csv")

print("Original Shape:", hate_df.shape)
hate_df.head()

Original Shape: (20000, 2)


,text,label
0,तुम्हें कुछ नहीं पता।,OFFENSIVE
1,धैर्य सफलता की कुंजी है।,NEUTRAL
2,धैर्य सफलता की कुंजी है।,NEUTRAL
3,अपना काम करो।,OFFENSIVE
4,अपना काम करो।,OFFENSIVE


In [4]:
hate_df = hate_df.rename(columns={
    "text": "text",
    "label": "hate_label"
})

In [5]:
hate_df.columns

Index(['text', 'hate_label'], dtype='object')

In [6]:
hate_df.isnull().sum()

,0
text,0
hate_label,0


In [7]:
hate_df = hate_df.dropna(subset=["text", "hate_label"])

print("After removing NaNs:", hate_df.shape)

After removing NaNs: (20000, 2)


In [8]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    return text

hate_df["text"] = hate_df["text"].apply(normalize_text)

In [9]:
def clean_text(text):
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)     # remove mentions
    text = re.sub(r"#\w+", "", text)     # remove hashtags
    text = re.sub(r"\s+", " ", text)     # remove extra spaces
    return text.strip()

hate_df["text"] = hate_df["text"].apply(clean_text)

In [10]:
# 🔹 Fixed Hate Label Mapping (Same as RoBERTa)
hate_label_map = {
    "HATE": 0,
    "OFFENSIVE": 1,
    "NEUTRAL": 2
}

# Clean label text (important for safety)
hate_df["hate_label"] = hate_df["hate_label"].astype(str).str.strip()

# Apply mapping
hate_df["hate_label"] = hate_df["hate_label"].map(hate_label_map)

# Check if any label failed mapping
if hate_df["hate_label"].isnull().sum() > 0:
    print("⚠️ Some hate labels not matching mapping!")
    print(hate_df[hate_df["hate_label"].isnull()])
else:
    print("✅ Hate labels mapped successfully.")

✅ Hate labels mapped successfully.


In [11]:
print(hate_df["hate_label"].value_counts())

hate_label
1    6697
2    6670
0    6633
Name: count, dtype: int64


In [12]:
hate_df = hate_df.reset_index(drop=True)

In [13]:
hate_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_hindi_hate.csv", index=False)

print("Cleaned Hate Dataset Saved.")

Cleaned Hate Dataset Saved.


# Emotion

In [14]:
emotion_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Raw/hindi_emotion_40k.csv")

print("Original Shape:", emotion_df.shape)
emotion_df.head()

Original Shape: (40000, 2)


,text,label
0,इस समय मुझे गुस्सा महसूस होता है आज। (0),anger
1,सच कहूँ तो मुझे बहुत नाराज़गी हो रही है आज। (1),anger
2,आज मुझे चिढ़ महसूस हो रही है ऐसी स्थिति में। (2),anger
3,सच कहूँ तो मुझे गुस्सा महसूस होता है इन हालात ...,anger
4,इस समय मुझे चिढ़ महसूस हो रही है इस समय। (4),anger


In [15]:
print("Columns:", emotion_df.columns)
print("\nSample Labels:")
print(emotion_df.iloc[:10])

Columns: Index(['text', 'label'], dtype='object')

Sample Labels:
                                                text  label
0           इस समय मुझे गुस्सा महसूस होता है आज। (0)  anger
1    सच कहूँ तो मुझे बहुत नाराज़गी हो रही है आज। (1)  anger
2   आज मुझे चिढ़ महसूस हो रही है ऐसी स्थिति में। (2)  anger
3  सच कहूँ तो मुझे गुस्सा महसूस होता है इन हालात ...  anger
4       इस समय मुझे चिढ़ महसूस हो रही है इस समय। (4)  anger
5  इस समय मुझे क्रोध महसूस हो रहा है ऐसी स्थिति म...  anger
6   सच कहूँ तो मुझे बहुत नाराज़गी हो रही है अभी। (6)  anger
7   आज मुझे चिढ़ महसूस हो रही है ऐसी स्थिति में। (7)  anger
8       इस समय मुझे बहुत गुस्सा आ रहा है इस समय। (8)  anger
9  इस समय मुझे बहुत नाराज़गी हो रही है इन हालात म...  anger


In [16]:
emotion_df = emotion_df.rename(columns={
    "text": "text",
    "label": "emotion"
})

In [17]:
emotion_df.columns

Index(['text', 'emotion'], dtype='object')

In [18]:
emotion_df = emotion_df.dropna(subset=["text","emotion"])

print("After removing nulls:", emotion_df.shape)

After removing nulls: (40000, 2)


In [19]:
import unicodedata

def normalize_text(text):
    return unicodedata.normalize("NFKC", str(text))

emotion_df["text"] = emotion_df["text"].apply(normalize_text)

In [20]:
import re

def clean_text(text):

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

emotion_df["text"] = emotion_df["text"].apply(clean_text)

In [21]:
emotion_df["emotion"] = (
    emotion_df["emotion"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [22]:
print("Cleaned unique labels:")
print(emotion_df["emotion"].unique())

Cleaned unique labels:
['anger' 'fear' 'hate' 'neutral' 'sadness']


In [23]:
emotion_label_map = {
    "anger":0,
    "hate":1,
    "sadness":2,
    "fear":3,
    "neutral":4
}

emotion_df["emotion_label"] = emotion_df["emotion"].map(emotion_label_map)

In [24]:
emotion_df["emotion_label"].value_counts()

,count
emotion_label,
0,8000
3,8000
1,8000
4,8000
2,8000


In [25]:
emotion_df = emotion_df[["text","emotion_label"]]

In [26]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
emotion_df["emotion_label"] = encoder.fit_transform(emotion_df["emotion_label"])

In [27]:
print(encoder.classes_)

[0 1 2 3 4]


In [28]:
print("Final encoded emotion labels:")
print(sorted(emotion_df["emotion_label"].unique()))

Final encoded emotion labels:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [29]:
emotion_df = emotion_df[["text","emotion_label"]]

In [30]:
emotion_df.head()

,text,emotion_label
0,इस समय मुझे गुस्सा महसूस होता है आज। (0),0
1,सच कहूँ तो मुझे बहुत नाराज़गी हो रही है आज। (1),0
2,आज मुझे चिढ़ महसूस हो रही है ऐसी स्थिति में। (2),0
3,सच कहूँ तो मुझे गुस्सा महसूस होता है इन हालात ...,0
4,इस समय मुझे चिढ़ महसूस हो रही है इस समय। (4),0


In [31]:
emotion_df["emotion_label"].value_counts()

,count
emotion_label,
0,8000
3,8000
1,8000
4,8000
2,8000


In [32]:
emotion_df = emotion_df.reset_index(drop=True)

In [33]:
emotion_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_hindi_emotion.csv", index=False)

print("✅ Cleaned Hindi Emotion Dataset Saved Successfully.")

✅ Cleaned Hindi Emotion Dataset Saved Successfully.


# Sentiment

In [34]:
import pandas as pd
import numpy as np
import unicodedata
import re

In [35]:
sentiment_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Raw/hindi_sentiment_dataset_20k.csv")

print("Original Shape:", sentiment_df.shape)
sentiment_df.head()

Original Shape: (20000, 2)


,text,label
0,शिक्षा जीवन बदल सकती है।,positive
1,अपना काम करो।,negative
2,शांत मन से सोचें।,positive
3,शांत मन से सोचें।,positive
4,आज का दिन अच्छा है।,positive


In [36]:
print("Columns:", sentiment_df.columns)

Columns: Index(['text', 'label'], dtype='object')


In [37]:
sentiment_df = sentiment_df.rename(columns={
    "text": "text",
    "label": "sentiment_label"
})

In [38]:
sentiment_df.columns

Index(['text', 'sentiment_label'], dtype='object')

In [39]:
sentiment_df = sentiment_df.dropna(subset=["text", "sentiment_label"])

print("After removing NaNs:", sentiment_df.shape)

After removing NaNs: (20000, 2)


In [40]:
def normalize_text(text):
    return unicodedata.normalize("NFKC", str(text))

sentiment_df["text"] = sentiment_df["text"].apply(normalize_text)

In [41]:
def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

sentiment_df["text"] = sentiment_df["text"].apply(clean_text)

In [42]:
print("Raw Sentiment Labels:")
print(sentiment_df["sentiment_label"].unique())

Raw Sentiment Labels:
['positive' 'negative' 'neutral']


In [43]:
sentiment_df["sentiment_label"] = (
    sentiment_df["sentiment_label"]
    .astype(str)
    .str.strip()
)

In [44]:
sentiment_label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

In [45]:
sentiment_df["sentiment_label"] = sentiment_df["sentiment_label"].map(sentiment_label_map)

In [46]:
if sentiment_df["sentiment_label"].isnull().sum() > 0:
    print("⚠️ Some sentiment labels not matching mapping!")
    print(sentiment_df[sentiment_df["sentiment_label"].isnull()].head())
else:
    print("✅ Sentiment labels mapped successfully.")

✅ Sentiment labels mapped successfully.


In [47]:
print("Final Encoded Sentiment Labels:")
print(sorted(sentiment_df["sentiment_label"].unique()))

Final Encoded Sentiment Labels:
[np.int64(0), np.int64(1), np.int64(2)]


In [48]:
sentiment_df = sentiment_df.reset_index(drop=True)

In [49]:
sentiment_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_hindi_sentiment.csv", index=False)

print("✅ Cleaned Hindi Sentiment Dataset Saved Successfully.")

✅ Cleaned Hindi Sentiment Dataset Saved Successfully.


# MULTI-TASK DATASET MERGING

In [50]:
hate_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_hindi_hate.csv")
emotion_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_hindi_emotion.csv")
sentiment_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/cleaned_hindi_sentiment.csv")

print("Hate:", hate_df.shape)
print("Emotion:", emotion_df.shape)
print("Sentiment:", sentiment_df.shape)

Hate: (20000, 2)
Emotion: (40000, 2)
Sentiment: (20000, 2)


In [51]:
hate_df["emotion_label"] = -100
hate_df["sentiment_label"] = -100

In [52]:
emotion_df["hate_label"] = -100
emotion_df["sentiment_label"] = -100

In [53]:
sentiment_df["hate_label"] = -100
sentiment_df["emotion_label"] = -100

In [54]:
hate_df = hate_df[["text", "hate_label", "emotion_label", "sentiment_label"]]
emotion_df = emotion_df[["text", "hate_label", "emotion_label", "sentiment_label"]]
sentiment_df = sentiment_df[["text", "hate_label", "emotion_label", "sentiment_label"]]

In [55]:
multi_df = pd.concat([hate_df, emotion_df, sentiment_df], axis=0)

multi_df = multi_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final Multi-task Dataset Shape:", multi_df.shape)
multi_df.head()

Final Multi-task Dataset Shape: (80000, 4)


,text,hate_label,emotion_label,sentiment_label
0,कभी कभी मुझे सब सामान्य है इन हालात में। (3044),-100,4,-100
1,कभी कभी मुझे सब सामान्य है इन हालात में। (295),-100,4,-100
2,शांत मन से सोचें।,-100,-100,2
3,तुम्हें कुछ नहीं पता।,-100,-100,0
4,मुझे सब सामान्य लग रहा है आज। (2645),-100,4,-100


In [56]:
print("Hate Labels Unique:", sorted(multi_df["hate_label"].unique()))
print("Emotion Labels Unique:", sorted(multi_df["emotion_label"].unique()))
print("Sentiment Labels Unique:", sorted(multi_df["sentiment_label"].unique()))

Hate Labels Unique: [np.int64(-100), np.int64(0), np.int64(1), np.int64(2)]
Emotion Labels Unique: [np.int64(-100), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Sentiment Labels Unique: [np.int64(-100), np.int64(0), np.int64(1), np.int64(2)]


In [57]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    multi_df,
    test_size=0.2,
    random_state=42
)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

Train Shape: (64000, 4)
Test Shape: (16000, 4)


In [58]:
train_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/hindi_multitask_train.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/hindi_multitask_test.csv", index=False)

print("✅ Hindi Multi-task Dataset Ready.")

✅ Hindi Multi-task Dataset Ready.


# **Training Phase**

In [59]:
!pip install huggingface_hub

In [60]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [61]:
!pip install transformers

In [62]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report

In [63]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [64]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/hindi_multitask_train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/EmiHate/Data/Cleaned/hindi_multitask_test.csv")

print(train_df.shape, test_df.shape)

(64000, 4) (16000, 4)


In [65]:
HF_TOKEN = "your hugging face token"

In [66]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "ai4bharat/indic-bert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
base_model = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)

print("IndicBERT loaded successfully!")

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IndicBERT loaded successfully!


In [67]:
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.hate_labels = df["hate_label"].tolist()
        self.emotion_labels = df["emotion_label"].tolist()
        self.sentiment_labels = df["sentiment_label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "hate_label": torch.tensor(self.hate_labels[idx], dtype=torch.long),
            "emotion_label": torch.tensor(self.emotion_labels[idx], dtype=torch.long),
            "sentiment_label": torch.tensor(self.sentiment_labels[idx], dtype=torch.long)
        }

In [68]:
train_dataset = MultiTaskDataset(train_df, tokenizer)
test_dataset  = MultiTaskDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [69]:
class MultiTaskIndicBERT(nn.Module):
    def __init__(self, encoder, hate_classes, emotion_classes, sentiment_classes):
        super().__init__()

        self.encoder = encoder
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        self.hate_head = nn.Linear(hidden_size, hate_classes)
        self.emotion_head = nn.Linear(hidden_size, emotion_classes)
        self.sentiment_head = nn.Linear(hidden_size, sentiment_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = torch.mean(outputs.last_hidden_state, dim=1)
        pooled = self.dropout(pooled)

        hate_logits = self.hate_head(pooled)
        emotion_logits = self.emotion_head(pooled)
        sentiment_logits = self.sentiment_head(pooled)

        return hate_logits, emotion_logits, sentiment_logits

In [70]:
hate_classes = 3
emotion_classes = 5
sentiment_classes = 3

model = MultiTaskIndicBERT(
    base_model,
    hate_classes,
    emotion_classes,
    sentiment_classes
)

model.to(device)

MultiTaskIndicBERT(
  (encoder): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(200000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertAttention(
                (attention_dropout): Dropout(p=0, inplace=False)
                (output_dropout): Dropout(p=0, inplace=False)
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Lin

In [71]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 1
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [72]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        hate_labels = batch["hate_label"].to(device)
        emotion_labels = batch["emotion_label"].to(device)
        sentiment_labels = batch["sentiment_label"].to(device)

        optimizer.zero_grad()

        hate_logits, emotion_logits, sentiment_logits = model(input_ids, attention_mask)

        loss = 0
        # Hate Loss
        if (hate_labels != -100).any():
          loss += criterion(hate_logits, hate_labels)

          # Emotion Loss
        if (emotion_labels != -100).any():
            loss += criterion(emotion_logits, emotion_labels)

          # Sentiment Loss
        if (sentiment_labels != -100).any():
            loss += criterion(sentiment_logits, sentiment_labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+10}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

KeyboardInterrupt: 

In [73]:
hate_label_map = {
    0: "Hate",
    1: "Offensive",
    2: "Neutral"
}

emotion_label_map = {
    0: "Anger",
    1: "Hate",
    2: "Sadness",
    3: "Fear",
    4: "Neutral"
}

sentiment_label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

In [74]:
import torch
import torch.nn.functional as F

def predict_text(text):
    model.eval()

    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        hate_logits, emotion_logits, sentiment_logits = model(input_ids, attention_mask)

    # Apply Softmax
    hate_probs = F.softmax(hate_logits, dim=1).cpu().numpy()[0]
    emotion_probs = F.softmax(emotion_logits, dim=1).cpu().numpy()[0]
    sentiment_probs = F.softmax(sentiment_logits, dim=1).cpu().numpy()[0]

    return hate_probs, emotion_probs, sentiment_probs

In [75]:
def display_prediction(text):
    hate_probs, emotion_probs, sentiment_probs = predict_text(text)

    print("📝 Input Text:")
    print(text)
    print("\n" + "="*60)

    # -------------------- HATE --------------------
    print("🚨 Hate Detection:")
    for i, prob in enumerate(hate_probs):
        print(f"{hate_label_map[i]}: {prob*100:.2f}%")

    hate_pred = hate_label_map[int(hate_probs.argmax())]
    print(f"\nFINAL HATE PREDICTION: {hate_pred}")

    print("\n" + "-"*60)

    # -------------------- EMOTION --------------------
    print("🧠 Top Emotions:")
    top2_indices = emotion_probs.argsort()[-2:][::-1]

    for idx in top2_indices:
        print(f"{emotion_label_map[idx]}: {emotion_probs[idx]*100:.2f}%")

    print("\n" + "-"*60)

    # -------------------- SENTIMENT --------------------
    print("📊 Sentiment:")
    for i, prob in enumerate(sentiment_probs):
        print(f"{sentiment_label_map[i]}: {prob*100:.2f}%")

    sentiment_pred = sentiment_label_map[int(sentiment_probs.argmax())]
    print(f"\nFINAL SENTIMENT: {sentiment_pred}")

    print("\n" + "="*60)

In [76]:
test_text = "तुम जैसे लोगों की वजह से सब खराब हो रहा है, बिल्कुल घटिया सोच है तुम्हारी 🤬 "

display_prediction(test_text)


📝 Input Text:
तुम जैसे लोगों की वजह से सब खराब हो रहा है, बिल्कुल घटिया सोच है तुम्हारी 🤬 

🚨 Hate Detection:
Hate: 36.00%
Offensive: 32.85%
Neutral: 31.15%

FINAL HATE PREDICTION: Hate

------------------------------------------------------------
🧠 Top Emotions:
Anger: 23.43%
Sadness: 20.79%

------------------------------------------------------------
📊 Sentiment:
Negative: 27.43%
Neutral: 36.68%
Positive: 35.89%

FINAL SENTIMENT: Neutral



In [77]:
test= "तुम लोग बिल्कुल बेकार हो 😡 तुम्हें कुछ भी नहीं आता 🤬"

display_prediction(test);

📝 Input Text:
तुम लोग बिल्कुल बेकार हो 😡 तुम्हें कुछ भी नहीं आता 🤬

🚨 Hate Detection:
Hate: 38.28%
Offensive: 31.78%
Neutral: 29.94%

FINAL HATE PREDICTION: Hate

------------------------------------------------------------
🧠 Top Emotions:
Anger: 24.67%
Hate: 20.95%

------------------------------------------------------------
📊 Sentiment:
Negative: 28.74%
Neutral: 34.41%
Positive: 36.85%

FINAL SENTIMENT: Positive



In [78]:
display_prediction("मुझे तुम पर बहुत गर्व है ❤️")

📝 Input Text:
मुझे तुम पर बहुत गर्व है ❤️

🚨 Hate Detection:
Hate: 40.41%
Offensive: 31.90%
Neutral: 27.70%

FINAL HATE PREDICTION: Hate

------------------------------------------------------------
🧠 Top Emotions:
Anger: 24.04%
Neutral: 21.60%

------------------------------------------------------------
📊 Sentiment:
Negative: 29.90%
Neutral: 33.34%
Positive: 36.76%

FINAL SENTIMENT: Positive



### Model Saving

In [79]:
save_path = "/content/drive/MyDrive/EmiHate/Models/indicbert_hin"

# Save encoder + tokenizer
model.encoder.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save full model weights (heads included)
import torch
torch.save(model.state_dict(), save_path + "/model_heads.pt")

print("✅ Hindi model saved locally")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Hindi model saved locally
